# VoxShield — speaker verification calibration

Phase 4. This does **not** train anything and does **not** need the anti-spoof
checkpoint or the waveform cache — it embeds a few thousand utterances with a
pretrained ECAPA-TDNN and measures where the decision thresholds should sit.

About ten minutes.

### Why it matters

`SpeakerThresholds` currently ships placeholders labelled `v0-UNCALIBRATED`.
There is no universal cosine threshold for "same speaker": a value tuned on
studio audio fails on a phone line. Quoting a decision rate from an
uncalibrated threshold is exactly what the handoff warns against in §48 and
§76. This replaces the guess with a measurement, and gives the speaker branch
its own EER — a number quotable the way 2.95 % is for anti-spoof.

### The split, and why it is not the obvious one

Calibrating and reporting on the *same* speakers is self-referential — the
speaker-verification version of quoting validation EER. So:

- **calibrate** on `validation.csv` (dev — 2,548 bonafide, 20 speakers)
- **measure** on `test.csv` (eval — 7,355 bonafide, **67 different speakers**)

The eval numbers then say how dev-derived thresholds behave on speakers they
were never tuned on, which is the only version of the question that matters.

### Before running

1. **Settings → Accelerator → GPU T4 ×2** (a P100 is sm_60 and will not run)
2. **Settings → Internet → On**
3. **Add Data → `asvpoof-2019-dataset-la`**

Nothing else. No previous notebook output needed.

## 1 · Environment

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

import torch

print("\ntorch", torch.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU. Settings > Accelerator > GPU T4 x2."

major = torch.cuda.get_device_capability()[0]
assert major >= 7, (
    f"sm_{major}x is too old for this PyTorch build. Switch to GPU T4 x2 - "
    "a P100 is sm_60 and its kernels are not compiled in."
)
print("gpu  ", torch.cuda.get_device_name(0))

# Kaggle's Internet setting is PER NOTEBOOK and defaults to off. It does not
# carry over from another notebook, which is the usual reason this fails.
import socket

socket.setdefaulttimeout(8)

online = True
try:
    socket.socket(socket.AF_INET, socket.SOCK_STREAM).connect(("pypi.org", 443))
except Exception as exc:
    online = False
    detail = f"{type(exc).__name__}: {exc}"

if online:
    print("internet: ON")
else:
    # Raised OUTSIDE the except block on purpose. Raising from inside one
    # chains the exceptions, and IPython's traceback formatter then fails on
    # the chain and buries this message under its own internal errors.
    print("=" * 62)
    print("  INTERNET IS OFF")
    print("=" * 62)
    print(f"  {detail}")
    print()
    print("  Right-hand panel -> Settings -> Internet -> On, then re-run.")
    print("  It needs phone verification on your Kaggle account.")
    print()
    print("  This notebook cannot run without it: speechbrain installs from")
    print("  PyPI and the ECAPA-TDNN weights download from HuggingFace.")
    print("=" * 62)
    raise RuntimeError("Enable Internet in Settings, then re-run this cell.")

## 2 · Code and dependencies

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

os.chdir("/kaggle/working")

WORK = Path("/kaggle/working/voxshield")
if WORK.exists():
    shutil.rmtree(WORK)

subprocess.run(
    ["git", "clone", "-q", "-b", "karthik", "https://github.com/SathvikGuttula/NullBox.git", str(WORK)], check=True
)
sys.path.insert(0, str(WORK / "backend"))

%cd /kaggle/working/voxshield
!git log --oneline -2

In [ ]:
# speechbrain brings the pretrained ECAPA-TDNN. soundfile reads the FLAC.
!pip install -q speechbrain soundfile

import importlib
for m in ["torch", "torchaudio", "speechbrain", "soundfile", "sklearn"]:
    try:
        print(f"{m:14}", importlib.import_module(m).__version__)
    except Exception as e:
        print(f"{m:14} MISSING  {type(e).__name__}")

## 3 · Manifests

Only the `speaker` column matters here. Bonafide utterances grouped by speaker
give target and nontarget pairs directly — no ASV protocol files needed.

In [ ]:
!python backend/scripts/build_manifest.py \
    --discover /kaggle/input/datasets \
    --manifest-dir /kaggle/working/voxshield/datasets/manifests \
    --holdout-attacks ""

In [ ]:
from collections import Counter
from app.ml.dataset import load_manifest

for name in ("validation.csv", "test.csv"):
    samples = load_manifest(f"datasets/manifests/{name}")
    bonafide = [s for s in samples if s.label == 0 and s.speaker]
    per_speaker = Counter(s.speaker for s in bonafide)

    print(f"{name:<16} {len(bonafide):>6} bonafide   "
          f"{len(per_speaker):>3} speakers   "
          f"{min(per_speaker.values())}-{max(per_speaker.values())} utterances each")

    assert len(per_speaker) >= 2, f"{name} has too few speakers to calibrate"

## 4 · Calibrate on dev speakers

Enrollment samples are held out of the probe pool. Scoring a probe against a
prototype that contains it would inflate target similarity towards 1.0, and the
resulting threshold would be one no live comparison can reach.

In [ ]:
!python backend/scripts/calibrate_speaker.py \
    --manifest datasets/manifests/validation.csv \
    --output /kaggle/working/speaker_thresholds_dev.json \
    --enroll-samples 5 \
    --max-probes 30 \
    --device cuda

## 5 · Measure on eval speakers — 67 speakers never seen during calibration

Two things come out of this cell:

- the **speaker EER on unseen speakers**, which is the number to report
- whether the dev thresholds still hit their targets here, which tells you
  whether a single threshold can serve both populations

In [ ]:
!python backend/scripts/calibrate_speaker.py \
    --manifest datasets/manifests/test.csv \
    --output /kaggle/working/speaker_thresholds_eval.json \
    --apply /kaggle/working/speaker_thresholds_dev.json \
    --enroll-samples 5 \
    --max-probes 20 \
    --device cuda

## 6 · Check the calibrated thresholds actually load

In [ ]:
import json
from app.ml.speaker import SpeakerRegistry, SpeakerThresholds

data = json.load(open("/kaggle/working/speaker_thresholds_dev.json"))

for name, mode in data["modes"].items():
    thresholds = SpeakerThresholds(**mode)
    print(f"{name:<16} match {thresholds.match:.4f}   "
          f"no_match {thresholds.no_match:.4f}   "
          f"calibrated={thresholds.calibrated}")
    assert thresholds.calibrated, "still flagged UNCALIBRATED"
    assert thresholds.no_match <= thresholds.match, "UNCERTAIN band inverted"

registry = SpeakerRegistry(SpeakerThresholds(**data["modes"]["balanced"]))
print(f"\nregistry ready, thresholds {registry.thresholds.version}")

## 7 · Summary and what to download

In [ ]:
import json
from pathlib import Path

W = Path("/kaggle/working")

print("=" * 66)
print("  SPEAKER VERIFICATION")
print("=" * 66)

for label, path in [("dev  (calibrated on)", W / "speaker_thresholds_dev.json"),
                    ("eval (unseen speakers)", W / "speaker_thresholds_eval.json")]:
    if path.exists():
        d = json.load(open(path))
        print(f"  {label:<24} EER {d['eer_percent']:6.2f} %   "
              f"{d['speakers']} speakers   "
              f"{d['targets'] + d['nontargets']:,} trials")
    else:
        print(f"  {label:<24} MISSING")

print("=" * 66)
print()
print("  DOWNLOAD (all tiny - a few KB each)")
for name in ("speaker_thresholds_dev.json", "speaker_thresholds_eval.json"):
    p = W / name
    mark = "[x]" if p.exists() else "[ ]"
    size = f"{p.stat().st_size/1024:.1f} KB" if p.exists() else "missing"
    print(f"    {mark} {size:>10}  {name}")
print()
print("  Send both to review. The eval EER is the number to report; the dev")
print("  thresholds are the ones you would deploy.")
print()
print("  These describe ASVspoof's clean studio audio. They will NOT transfer")
print("  to telephone conditions - re-calibrate before quoting them for a")
print("  phone deployment.")
print()